In [1]:
import gc
gc.collect()

75

In [3]:
from core.file_manager import preprocess_file_manager
from core.visualization_lib import folder_shower, normalize_volume
from core.helper import  copy_NiFty, patients_transform
from core.helper import transform_step_list_to_dictioanry

from core.transformers.nifti_to_raw_transformer import nifti_to_raw_transformer
from core.transformers.anatomy_fill_transformer import anatomy_fill_transformer
from core.transformers.crop_transformer import non_weighted_crop_transformer
from core.transformers.resample_transformer import resample_transformer


In [4]:
original_data_folder =  '/home/robakp/Exeriments1/prostate_lesion_detection/rjozwiak-MGR_dataset_correct/MGR_dataset_correct'

channels = {
    'adc' : 'adc',
    'anatomy' : 'anatomy',
    'dwi' : 'dwi',
    't2' : 't2'
}

target = 'lesion'

file_extention = '.nii.gz'

preprocessing_steps_list = [
    ('start', 'nifty'),
    ('resampling','resampled'),
    ('nifti_to_raw', 'raw'),
    ('filling_anatomy_gaps', 'anatomy_gap_filled'),
    ('cropping', 'cropped'),

]

preprocessed_steps = transform_step_list_to_dictioanry(preprocessing_steps_list)

crop_size = (160,160,24)
target_spacing=(0.8, 0.8, 3.5)

In [5]:
file_manager = preprocess_file_manager('/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed',preprocessed_steps,channels)

In [6]:
preprocessed_steps

{'resampling': {'start': '0_nifty', 'end': '1_resampled'},
 'nifti_to_raw': {'start': '1_resampled', 'end': '2_raw'},
 'filling_anatomy_gaps': {'start': '2_raw', 'end': '3_anatomy_gap_filled'},
 'cropping': {'start': '3_anatomy_gap_filled', 'end': '4_cropped'}}

copy data to preprocess folder

In [7]:
copy_NiFty(original_data_folder, file_manager,channels, filter=['3322','001','003','004'], step=preprocessed_steps[preprocessing_steps_list[1][0]]['start'])
patients = file_manager.get_file_names()

<h2>resampling</h2>


In [23]:
resample_transformer = resample_transformer(target_spacing=target_spacing,crop_size=crop_size, is_label=False)


start_step = preprocessed_steps['resampling']['start']
end_step = preprocessed_steps['resampling']['end']
patients_transform(file_manager, start_step, end_step,resample_transformer)


transform to raw

In [24]:
start_step = preprocessed_steps['nifti_to_raw']['start']
end_step = preprocessed_steps['nifti_to_raw']['end']

to_raw_transform = nifti_to_raw_transformer()

patients_transform(file_manager, start_step, end_step, to_raw_transform)

dispaly

In [25]:
folder_shower(file_manager,'2_raw',normalize_volume)

interactive(children=(Dropdown(description='Patient:', options=('001', '003', '004', '3322'), value='001'), Dr…

remember about allingning

<h2>Working with holes in prostate layer</h2>

In [ ]:
start_step = preprocessed_steps['filling_anatomy_gaps']['start']
end_step = preprocessed_steps['filling_anatomy_gaps']['end']

checking out outliers

FIX outliers

In [28]:
gap_fill_transformer = anatomy_fill_transformer()
patients_transform(file_manager, start_step, end_step,gap_fill_transformer)

19  21


<h2>CROPPING!!!</h2>

Cutting pictures into correct sizes

two ways of centering

finding maximum prostate dimentions

In [29]:
from core.stat_calc import find_patients_max_prostate_sizes

start_step = preprocessed_steps['cropping']['start']
end_step = preprocessed_steps['cropping']['end']

maximum_prostate_size = find_patients_max_prostate_sizes(patients,file_manager, step = start_step)
print(maximum_prostate_size)

[67, 60, 15]


center cropping

THERE IS NO PADDING!!!!

In [30]:
nw_crop_transformer = non_weighted_crop_transformer(crop_size)
patients_transform(file_manager, start_step, end_step, nw_crop_transformer)

In [31]:
folder_shower(file_manager,normalizer=normalize_volume,step = start_step)
folder_shower(file_manager,step = end_step,normalizer=normalize_volume)

interactive(children=(Dropdown(description='Patient:', options=('001', '003', '004', '3322'), value='001'), Dr…

interactive(children=(Dropdown(description='Patient:', options=('001', '003', '004', '3322'), value='001'), Dr…